# Populacja i cechy związane z udarem mózgu

Ten notebook jest liniowym raportem opisowym dla śledzonego zbioru *Stroke Prediction Dataset*. Zaczynamy od całej populacji, sprawdzamy pochodzenie i jakość danych, a następnie opisujemy rozkłady oraz różnice między obserwacjami z udarem i bez udaru. Dopiero po zakończeniu części populacyjnej uruchamiamy wspólny pipeline modelowania.

**Zakres interpretacji:** wykresy i statystyki opisują współwystępowanie cech w tym zbiorze. Nie są dowodem przyczynowości ani skuteczności interwencji medycznych. Wynik modelu jest predykcją akademicką, a nie diagnozą.


In [ ]:
# Inicjalizacja środowiska i wspólne definicje projektu
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# Działa zarówno po uruchomieniu z katalogu projektu, jak i z notebooks/.
def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src").is_dir() and (candidate / "data" / "raw").is_dir():
            return candidate
    raise RuntimeError("Nie znaleziono katalogu projektu z src/ i data/raw/.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (  # noqa: E402
    BINARY_COLS,
    CATEGORICAL_COLS,
    LOCAL_RAW_DATA_PATH,
    NUMERIC_CONTINUOUS_COLS,
    TARGET_COL,
)
from src.data import curate_stroke_data, load_data, validate_missing_bmi_patterns  # noqa: E402
from src.ml import predict_stroke_risk, run_ml_benchmark  # noqa: E402

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titleweight"] = "bold"

print(f"Katalog projektu: {PROJECT_ROOT}")
print(f"Śledzony plik danych: {LOCAL_RAW_DATA_PATH.relative_to(PROJECT_ROOT)}")


## 1. Pochodzenie danych i definicja populacji

Źródłem analiz jest lokalny, śledzony plik `data/raw/healthcare-dataset-stroke-data.csv`. Oryginalny *Stroke Prediction Dataset* jest publikowany jako [fedesoriano/stroke-prediction-dataset na Kaggle](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset); repozytorium zawiera lokalną kopię pliku, aby wykonanie było powtarzalne. Przed dalszym użyciem danych poza tym projektem należy sprawdzić warunki źródłowej licencji i atrybucji. Funkcja `src.data.load_data()` jest jedynym wejściem danych: sprawdza wymagane kolumny, nie pobiera plików zdalnie i kończy wykonanie czytelnym błędem, jeśli pliku brakuje lub jest niepoprawny. Nazwy kolumn, cechy kategoryczne, binarne i ciągłe pochodzą z `src.config`, czyli z tej samej definicji, której używa aplikacyjny pipeline.

W części opisowej wykorzystujemy pełną populację z pliku. Dla modelowania `curate_stroke_data()` tworzy kohortę dorosłych (`age >= 18`) i osobną kohortę pediatryczną; pojedyncza obserwacja `gender == "Other"` jest usuwana wyłącznie z kohorty modelowej zgodnie z decyzją pipeline'u. Brakujące BMI pozostaje jawne w analizie jakości, a pipeline modelowy imputuje je wewnątrz przygotowania danych.


In [ ]:
# Wczytanie i walidacja śledzonego zbioru
raw_df = load_data()

print(f"Kształt zbioru: {raw_df.shape[0]:,} wierszy × {raw_df.shape[1]} kolumn")
print(f"Zakres wieku: {raw_df['age'].min():.0f}–{raw_df['age'].max():.0f} lat")
print(f"Kolumny: {', '.join(raw_df.columns)}")

quality_summary = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "braki (n)": raw_df.isna().sum(),
    "braki (%)": (raw_df.isna().mean() * 100).round(2),
    "unikalne wartości": raw_df.nunique(dropna=True),
})
display(quality_summary)


### Decyzje jakościowe

Walidacja potwierdza niepusty zbiór, unikalne identyfikatory, binarny i kompletny cel `stroke` oraz obecność dorosłych i dzieci. Braki występują przede wszystkim w `bmi`; nie usuwamy tych wierszy, ponieważ zmniejszyłoby to badaną populację. W modelowaniu brakujące wartości są obsługiwane przez wspólny preprocesor: mediana dla liczb oraz najczęstsza kategoria dla zmiennych kategorycznych.


In [ ]:
# Jawne podsumowanie braków BMI według kohorty i wyniku
bmi_quality = validate_missing_bmi_patterns(raw_df)
display(bmi_quality)


In [ ]:
# Wykres 1/7 — liczebność i niezrównoważenie celu
outcome_counts = raw_df[TARGET_COL].value_counts().sort_index().rename(index={0: "Bez udaru", 1: "Udar"})
outcome_percent = outcome_counts / outcome_counts.sum() * 100

fig, ax = plt.subplots(figsize=(7, 4.5))
colors = ["#94a3b8", "#dc2626"]
bars = ax.bar(outcome_counts.index, outcome_counts.values, color=colors, width=0.62)
for bar, count, percent in zip(bars, outcome_counts.values, outcome_percent.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 55,
            f"{count:,} ({percent:.1f}%)", ha="center", va="bottom", fontweight="bold")
ax.set_title("Wynik obserwacji w całej populacji")
ax.set_ylabel("Liczba obserwacji")
ax.set_ylim(0, outcome_counts.max() * 1.16)
plt.tight_layout()
plt.show()

print(f"Udar stwierdzono w {int(outcome_counts.loc['Udar']):,} z {len(raw_df):,} obserwacji ({outcome_percent.loc['Udar']:.2f}%).")


## 2. Rozkłady cech w populacji

Poniższe wykresy pokazują, jak wygląda badana populacja przed jakimkolwiek porównaniem grup. Są to rozkłady obserwowanych wartości, a nie rozkłady populacji ogólnej w sensie epidemiologicznym; reprezentatywność zależy od sposobu zebrania źródłowego zbioru.


In [ ]:
# Wykres 2/7 — rozkłady głównych zmiennych liczbowych
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
numeric_labels = {
    "age": "Wiek (lata)",
    "avg_glucose_level": "Średni poziom glukozy",
    "bmi": "BMI",
}
for ax, column in zip(axes, NUMERIC_CONTINUOUS_COLS):
    sns.histplot(raw_df[column], bins=30, kde=True, color="#2563eb", ax=ax)
    ax.set_title(numeric_labels[column])
    ax.set_xlabel(numeric_labels[column])
    ax.set_ylabel("Liczba obserwacji")
fig.suptitle("Rozkłady zmiennych liczbowych", y=1.03, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Wykres 3/7 — skład populacji według cech kategorycznych
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, column in zip(axes.flat, CATEGORICAL_COLS):
    order = raw_df[column].value_counts().index
    sns.countplot(data=raw_df, x=column, order=order, color="#0f766e", ax=ax)
    ax.set_title(column)
    ax.set_xlabel("")
    ax.set_ylabel("Liczba obserwacji")
    ax.tick_params(axis="x", rotation=35)
axes.flat[-1].axis("off")
fig.suptitle("Skład populacji według zmiennych kategorycznych", y=1.01, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Wykres 4/7 — profil braków danych
missing = raw_df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
fig, ax = plt.subplots(figsize=(7, 4.2))
if missing.empty:
    ax.text(0.5, 0.5, "Brak brakujących wartości", ha="center", va="center")
    ax.set_axis_off()
else:
    bars = ax.bar(missing.index, missing.values, color="#f59e0b")
    ax.set_ylabel("Liczba braków")
    ax.set_title("Braki danych wymagające jawnej decyzji")
    ax.tick_params(axis="x", rotation=30)
    for bar, value in zip(bars, missing.values):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 3, str(value), ha="center", va="bottom")
plt.tight_layout()
plt.show()


## 3. Porównanie obserwacji z udarem i bez udaru

Porównania poniżej są opisowe: pokazują różnice w rozkładach i odsetkach w tym zbiorze, ale nie kontrolują wszystkich innych cech i nie ustalają kierunku ani przyczynowości zależności. Liczebność grup jest bardzo różna, dlatego surowe liczby i odsetki należy czytać razem.


In [ ]:
# Przygotowanie etykiet i kohort używanych w dalszej części
analysis_df = raw_df.copy()
analysis_df["status"] = analysis_df[TARGET_COL].map({0: "Bez udaru", 1: "Udar"})
cohorts = curate_stroke_data(raw_df, remove_other_gender=True, age_threshold=18.0)
adult_df = cohorts.adult
paediatric_df = cohorts.paediatric

cohort_table = pd.DataFrame({
    "Kohorta": ["Cała populacja", "Dorośli (wiek >= 18)", "Pediatryczna (wiek < 18)"],
    "N": [len(raw_df), len(adult_df), len(paediatric_df)],
    "Udar (n)": [int(raw_df[TARGET_COL].sum()), int(adult_df[TARGET_COL].sum()), int(paediatric_df[TARGET_COL].sum())],
})
cohort_table["Udar (%)"] = (cohort_table["Udar (n)"] / cohort_table["N"] * 100).round(2)
display(cohort_table)


In [ ]:
# Wykres 5/7 — różnica wieku między grupami wyniku
fig, ax = plt.subplots(figsize=(8, 4.8))
sns.violinplot(data=analysis_df, x="status", y="age", order=["Bez udaru", "Udar"],
               inner=None, cut=0, palette={"Bez udaru": "#94a3b8", "Udar": "#dc2626"}, ax=ax)
sns.boxplot(data=analysis_df, x="status", y="age", order=["Bez udaru", "Udar"],
            width=0.18, showfliers=False, color="white", linewidth=1.4, ax=ax)
ax.set_title("Wiek w grupach z udarem i bez udaru")
ax.set_xlabel("")
ax.set_ylabel("Wiek (lata)")
plt.tight_layout()
plt.show()

age_summary = analysis_df.groupby("status")["age"].agg(["count", "median", "mean"]).reindex(["Bez udaru", "Udar"]).round(1)
display(age_summary)


In [ ]:
# Wykres 6/7 — porównanie glukozy i BMI według wyniku
comparison_long = analysis_df.melt(
    id_vars="status", value_vars=["avg_glucose_level", "bmi"],
    var_name="cecha", value_name="wartość"
).dropna()
comparison_long["cecha"] = comparison_long["cecha"].map({
    "avg_glucose_level": "Glukoza",
    "bmi": "BMI",
})
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, feature in zip(axes, ["Glukoza", "BMI"]):
    subset = comparison_long[comparison_long["cecha"] == feature]
    sns.boxenplot(data=subset, x="status", y="wartość", order=["Bez udaru", "Udar"],
                  palette={"Bez udaru": "#94a3b8", "Udar": "#dc2626"}, ax=ax)
    ax.set_title(feature)
    ax.set_xlabel("")
    ax.set_ylabel("Wartość")
fig.suptitle("Rozkład glukozy i BMI według wyniku", y=1.02, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Wykres 7/7 — relacja wieku i glukozy z oznaczeniem wyniku
scatter_df = analysis_df.dropna(subset=["age", "avg_glucose_level"]).sample(
    n=min(2500, len(analysis_df)), random_state=RANDOM_STATE
)
fig, ax = plt.subplots(figsize=(8.5, 5.2))
sns.scatterplot(
    data=scatter_df, x="age", y="avg_glucose_level", hue="status",
    hue_order=["Bez udaru", "Udar"], palette={"Bez udaru": "#64748b", "Udar": "#dc2626"},
    alpha=0.45, s=35, linewidth=0, ax=ax,
)
ax.set_title("Wiek i poziom glukozy według wyniku")
ax.set_xlabel("Wiek (lata)")
ax.set_ylabel("Średni poziom glukozy")
ax.legend(title="Wynik")
plt.tight_layout()
plt.show()


## 4. Descriptive interpretation

W tym zbiorze udar jest klasą mniejszościową, a obserwacje z udarem mają inny opisowy profil wieku niż obserwacje bez udaru. Rozkłady glukozy i BMI oraz skład cech kategorycznych również różnią się między grupami, ale ich nakładanie się pokazuje, że żadna pojedyncza cecha nie rozdziela obserwacji idealnie. Wykres wieku i glukozy pokazuje współwystępowanie dwóch zmiennych, nie efekt przyczynowy.

Te wyniki służą do zrozumienia kohorty i przygotowania modelowania. Nie należy interpretować różnic jako dowodu, że zmiana konkretnej cechy spowoduje zmianę ryzyka udaru; do tego potrzebne byłyby inne dane i metody przyczynowe.


## 5. Przejście do wspólnego pipeline'u modelowania

Dopiero po zakończeniu opisu populacji używamy kohorty dorosłych i `src.ml.run_ml_benchmark()`. Ta sama definicja cech z `src.config` jest używana przez pipeline; modele korzystają z jednego, stratyfikowanego podziału, imputacji i kodowania. Wyniki poniżej są predykcyjne i nie zmieniają opisowego charakteru wcześniejszych obserwacji.


In [ ]:
# Benchmark wspólnego pipeline'u: Logistic Regression i Random Forest
benchmark = run_ml_benchmark(adult_df, test_size=0.20, random_state=RANDOM_STATE)
model_results = benchmark["results"].copy()
model_results["threshold"] = model_results["threshold"].round(3)
model_results[["model", "threshold", "f1", "precision", "recall"]]


In [ ]:
# Przykład predykcji jednej osoby — wynik nie jest diagnozą
patient_sample = {
    "gender": "Female",
    "age": 55.0,
    "hypertension": 0,
    "heart_disease": 0,
    "ever_married": "Yes",
    "work_type": "Private",
    "Residence_type": "Urban",
    "avg_glucose_level": 110.0,
    "bmi": 28.0,
    "smoking_status": "never smoked",
}

prediction = predict_stroke_risk(benchmark, patient_sample)
prediction_summary = pd.DataFrame([{
    "Prawdopodobieństwo predykcji (%)": round(prediction["probability_percent"], 2),
    "Próg F1": round(prediction["decision_threshold"], 3),
    "Klasyfikacja": "powyżej progu" if prediction["screening_positive"] else "poniżej progu",
    "Uwaga": "Nie jest to diagnoza medyczna",
}])
display(prediction_summary)


## Podsumowanie

Notebook jest wykonywalnym, liniowym przejściem od śledzonego źródła danych przez jakość i opis populacji do porównań wyniku, a następnie do wspólnego pipeline'u predykcyjnego. Siedem wykresów odpowiada różnym pytaniom: jak liczna i niezrównoważona jest kohorta, jakie są rozkłady liczbowe i kategoryczne, gdzie występują braki, jak różnią się grupy wyniku oraz jak współwystępują ważne cechy. Wszystkie wnioski populacyjne pozostają opisowe i nie-causalne.
